# Visualización de Resultados: SBOMs y Vulnerabilidades

Este notebook se encarga de leer los resultados (archivos JSON) obtenidos del descubrimiento de dependencias y escaneo de vulnerabilidades, para presentar un resumen estadístico e ilustrativo.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Análisis de Dependencias (SBOMs)
Cargaremos los datos de todos los SBOMs generados en la carpeta `data/sboms`.

In [ ]:
deps_data = []
for file in os.listdir("data/sboms"):
    if file.endswith(".json"):
        repo = file.replace(".json", "")
        with open(f"data/sboms/{file}", "r") as f:
            try:
                data = json.load(f)
                artifacts = data.get("artifacts", [])
                for a in artifacts:
                    deps_data.append({
                        "repo": repo,
                        "name": a.get("name"),
                        "version": a.get("version"),
                        "type": a.get("type"),
                        "language": a.get("language")
                    })
            except Exception as e:
                print(f"Error procesando {file}: {e}")

df_deps = pd.DataFrame(deps_data)
print(f"Total de dependencias procesadas: {len(df_deps)}")
df_deps.head()

In [ ]:
if not df_deps.empty:
    plt.figure(figsize=(10, 6))
    ecosystem_counts = df_deps['type'].value_counts()
    sns.barplot(x=ecosystem_counts.values, y=ecosystem_counts.index, palette="viridis")
    plt.title("Distribución de Ecosistemas de Paquetes en 'openclaw'")
    plt.xlabel("Cantidad de dependencias encontradas")
    plt.ylabel("Ecosistema (Tipo)")
    plt.tight_layout()
    plt.show()

In [ ]:
if not df_deps.empty:
    plt.figure(figsize=(10, 6))
    repo_counts = df_deps['repo'].value_counts()
    sns.barplot(x=repo_counts.values, y=repo_counts.index, palette="mako")
    plt.title("Cantidad de dependencias por Repositorio")
    plt.xlabel("Cantidad de dependencias")
    plt.ylabel("Repositorio")
    plt.tight_layout()
    plt.show()

## 2. Análisis de Vulnerabilidades
Cargaremos los datos de todos los reportes generados en la carpeta `data/vulnerabilidades`.

In [ ]:
vuln_data = []
for file in os.listdir("data/vulnerabilidades"):
    if file.endswith(".json"):
        repo = file.replace(".json", "")
        with open(f"data/vulnerabilidades/{file}", "r") as f:
            try:
                data = json.load(f)
                matches = data.get("matches", [])
                for m in matches:
                    vuln = m.get("vulnerability", {})
                    artifact = m.get("artifact", {})
                    vuln_data.append({
                        "repo": repo,
                        "vulnerability_id": vuln.get("id"),
                        "severity": vuln.get("severity"),
                        "package_name": artifact.get("name"),
                        "package_version": artifact.get("version"),
                        "type": artifact.get("type")
                    })
            except Exception as e:
                print(f"Error procesando {file}: {e}")

df_vulns = pd.DataFrame(vuln_data)
if not df_vulns.empty:
    print(f"Total de vulnerabilidades encontradas: {len(df_vulns)}")
    display(df_vulns.head())
else:
    print("No se encontraron vulnerabilidades.")

In [ ]:
if not df_vulns.empty:
    plt.figure(figsize=(8, 5))
    severity_order = ["Critical", "High", "Medium", "Low", "Unknown", "Negligible"]
    valid_severities = [s for s in severity_order if s in df_vulns['severity'].unique()]
    
    color_map = {
        "Critical": "#8b0000",
        "High": "#d32f2f",
        "Medium": "#f57c00",
        "Low": "#fbc02d",
        "Unknown": "#757575",
        "Negligible": "#bdbdbd"
    }
    
    sns.countplot(data=df_vulns, x='severity', order=valid_severities, palette=color_map)
    plt.title("Conteo de Vulnerabilidades por Nivel de Gravedad")
    plt.xlabel("Gravedad (Severity)")
    plt.ylabel("Cantidad de Vulnerabilidades")
    plt.show()

In [ ]:
if not df_vulns.empty:
    plt.figure(figsize=(10, 6))
    top_vuln_packages = df_vulns['package_name'].value_counts().head(10)
    sns.barplot(x=top_vuln_packages.values, y=top_vuln_packages.index, palette="rocket")
    plt.title("Top 10 Paquetes con Mayor Cantidad de Vulnerabilidades")
    plt.xlabel("Número de vulnerabilidades reportadas")
    plt.ylabel("Nombre del Paquete")
    plt.tight_layout()
    plt.show()

In [ ]:
if not df_vulns.empty:
    plt.figure(figsize=(10, 6))
    repo_vuln_counts = df_vulns['repo'].value_counts()
    sns.barplot(x=repo_vuln_counts.values, y=repo_vuln_counts.index, palette="magma")
    plt.title("Cantidad de Vulnerabilidades por Repositorio")
    plt.xlabel("Número de vulnerabilidades")
    plt.ylabel("Repositorio")
    plt.tight_layout()
    plt.show()